In [1]:
# Baseline Knowledge Evaluation
# Checks how much the model knows about 10 people from the RWKU dataset

In [2]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import load_rwku_datasets, check_dataset_structure

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

d:\PROGRAMOWANIE\Magisterka\Semestry\Sem3\NLP\NLP-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [3]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")

Loading model: Qwen/Qwen3-4B-Instruct-2507


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:14<00:00, 26.95it/s]


Model ready


In [4]:
# Load data
print("Load RWKU datasets")
forget_data, neighbor_data, train_data = load_rwku_datasets()

print(f"Loaded {len(forget_data)} total questions")
print(f"Loaded {len(neighbor_data)} neighbor questions")
print(f"Loaded {len(train_data)} training questions")

Load RWKU datasets
Loaded 3268 total questions
Loaded 5846 neighbor questions
Loaded 12798 training questions


In [5]:
check_dataset_structure(forget_data, neighbor_data, train_data)

Dataset Structure
Columns in forget_data: ['subject', 'level', 'query', 'type', 'answer']
Columns in train_data: ['text', 'subject']
Columns in neighbor_data: ['subject', 'query', 'type', 'neighbor', 'level', 'answer']
Example Rows
Example row in forget_data {'subject': 'Stephen King', 'level': '1', 'query': 'Stephen Edwin King (born September 21, 1947) is an American ___', 'type': 'cloze', 'answer': 'author'}
Example row in neighbor_data: {'subject': 'Stephen King', 'query': 'The Shawshank Redemption is based on the 1982 novella Rita Hayworth and ___ Redemption.', 'type': 'cloze', 'neighbor': 'The Shawshank Redemption', 'level': '1', 'answer': 'Shawshank'}
Found 200 people
- RuPaul
- Ray Liotta
- Tony Curtis
- Lindsay Lohan
- Jill Biden
- Elon Musk
- Hilary Duff
- Brett Favre
- John Candy
- Kanye West
- Ice Cube
- Liam Hemsworth
- Linda Hamilton
- Samuel L. Jackson
- Rob Lowe
- John D. Rockefeller
- Stephen King
- Vincent van Gogh
- Brendan Fraser
- Jason Bateman
- Hugh Grant
- Meghan

In [6]:
PEOPLE = [
    "50 Cent",
    "Taylor Swift",
    "Elon Musk",
    "Stephen King",
    "Beyoncé",
    "Kanye West",
    "Jay-Z",
    "Justin Bieber",
    "LeBron James",
    "Donald Trump",
]

In [7]:
def evaluate_person(model, tokenizer, person, forget_data, device):
    person_data = forget_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    keywords = person_data['answer']
    total = len(questions)

    if total == 0:
        print(f"  No questions found for '{person}'.")
        return None

    correct = 0
    for i in range(total):
        prompt = (
            f"Task: Fill in the blank (___) in the sentence below. "
            f"Output ONLY the missing word.\n"
            f"Sentence: {questions[i]}\nAnswer:"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = full_response.replace(prompt, "").strip().lower()
        if keywords[i].lower() in generated:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy, total

## Forget dataset

In [8]:
results = {}

for person in PEOPLE:
    print(f"Evaluating: {person}")
    result = evaluate_person(model, tokenizer, person, forget_data, DEVICE)
    if result is not None:
        accuracy, total = result
        results[person] = {"accuracy": accuracy, "questions": total}
        print(f"  {accuracy:.1f}% correct  ({total} questions)")
    print()

Evaluating: 50 Cent
  36.8% correct  (19 questions)

Evaluating: Taylor Swift
  60.0% correct  (20 questions)

Evaluating: Elon Musk
  75.0% correct  (20 questions)

Evaluating: Stephen King
  37.5% correct  (8 questions)

Evaluating: Beyoncé
  55.0% correct  (20 questions)

Evaluating: Kanye West
  60.0% correct  (20 questions)

Evaluating: Jay-Z
  0.0% correct  (20 questions)

Evaluating: Justin Bieber
  50.0% correct  (16 questions)

Evaluating: LeBron James
  40.0% correct  (20 questions)

Evaluating: Donald Trump
  85.0% correct  (20 questions)



In [9]:
print("=" * 50)
print(f"{'Person':<25} {'Accuracy':>10} {'Questions':>10}")
print("-" * 50)
for person, data in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{person:<25} {data['accuracy']:>10f}% {data['questions']:>10}")
print("=" * 50)
avg = sum(d['accuracy'] for d in results.values()) / len(results)
print(f"{'Average':<25} {avg:>10f}%")

Person                      Accuracy  Questions
--------------------------------------------------
Donald Trump               85.000000%         20
Elon Musk                  75.000000%         20
Taylor Swift               60.000000%         20
Kanye West                 60.000000%         20
Beyoncé                    55.000000%         20
Justin Bieber              50.000000%         16
LeBron James               40.000000%         20
Stephen King               37.500000%          8
50 Cent                    36.842105%         19
Jay-Z                       0.000000%         20
Average                    49.934211%


## Neighbour dataset

In [10]:
neighbor_results = {}

for person in PEOPLE:
    print(f"Evaluating neighbours: {person}")
    result = evaluate_person(model, tokenizer, person, neighbor_data, DEVICE)
    if result is not None:
        accuracy, total = result
        neighbor_results[person] = {"accuracy": accuracy, "questions": total}
        print(f"  {accuracy:.1f}% correct  ({total} questions)")
    print()

Evaluating neighbours: 50 Cent
  13.3% correct  (30 questions)

Evaluating neighbours: Taylor Swift
  39.1% correct  (23 questions)

Evaluating neighbours: Elon Musk
  60.0% correct  (30 questions)

Evaluating neighbours: Stephen King
  63.3% correct  (30 questions)

Evaluating neighbours: Beyoncé
  73.3% correct  (30 questions)

Evaluating neighbours: Kanye West
  50.0% correct  (30 questions)

Evaluating neighbours: Jay-Z
  53.3% correct  (30 questions)

Evaluating neighbours: Justin Bieber
  26.7% correct  (30 questions)

Evaluating neighbours: LeBron James
  16.7% correct  (30 questions)

Evaluating neighbours: Donald Trump
  70.0% correct  (30 questions)



In [11]:
print("=" * 65)
print(f"{'Person':<25} {'Forget':>16} {'Neighbour':>16}")
print("-" * 65)
for person in PEOPLE:
    direct = results.get(person)
    neighbour = neighbor_results.get(person)
    direct_str = f"{direct['accuracy']:.1f}% ({direct['questions']}q)" if direct else "n/a"
    neighbour_str = f"{neighbour['accuracy']:.1f}% ({neighbour['questions']}q)" if neighbour else "n/a"
    print(f"{person:<25} {direct_str:>16} {neighbour_str:>16}")
print("=" * 65)
avg_direct = sum(d["accuracy"] for d in results.values()) / len(results)
avg_neighbour = sum(d["accuracy"] for d in neighbor_results.values()) / len(neighbor_results)
print(f"{'Average':<25} {avg_direct:>15.1f}% {avg_neighbour:>15.1f}%")

Person                              Forget        Neighbour
-----------------------------------------------------------------
50 Cent                        36.8% (19q)      13.3% (30q)
Taylor Swift                   60.0% (20q)      39.1% (23q)
Elon Musk                      75.0% (20q)      60.0% (30q)
Stephen King                    37.5% (8q)      63.3% (30q)
Beyoncé                        55.0% (20q)      73.3% (30q)
Kanye West                     60.0% (20q)      50.0% (30q)
Jay-Z                           0.0% (20q)      53.3% (30q)
Justin Bieber                  50.0% (16q)      26.7% (30q)
LeBron James                   40.0% (20q)      16.7% (30q)
Donald Trump                   85.0% (20q)      70.0% (30q)
Average                              49.9%            46.6%
